In [1]:
import os,sys,time
print("=== CROSSWALK V3 BOOT ===",flush=True)
print("[BOOT] Python:",sys.version,flush=True)
print("[BOOT] heartbeat 1",flush=True); time.sleep(1)
print("[BOOT] heartbeat 2",flush=True)
open("/kaggle/working/CROSSWALK_V3_STARTED.txt","w").write("started\n")
print("=== CROSSWALK V3 HEARTBEAT OK ===",flush=True)

=== CROSSWALK V3 BOOT ===
[BOOT] Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
[BOOT] heartbeat 1
[BOOT] heartbeat 2
=== CROSSWALK V3 HEARTBEAT OK ===


# 8% — Crosswalk Safety v3

Improved full-video safety-zone conflict screening.

**Upgrade dari v2**
- tetap proses seluruh 120 video / 3 part;
- sampling dinaikkan dari 5 detik → **2 detik**;
- bedakan **vehicle presence**, **pedestrian presence**, dan **conflict frame**;
- conflict = kendaraan dan pedestrian sama-sama berada di demo ROI pada waktu sampel yang sama;
- frame conflict yang berdekatan digabung menjadi **temporal conflict event** agar tidak dihitung berulang;
- tetap jujur: ROI adalah **demo safety-zone proxy**, bukan ground-truth zebra crossing.

In [2]:
from pathlib import Path
ROOTS=[
    Path("/kaggle/input/datasets/loliwibu/crosswalkpart-1"),
    Path("/kaggle/input/datasets/loliwibu/crosswalk-part-2"),
    Path("/kaggle/input/datasets/loliwibu/crosswalk-part-3"),
]
for r in ROOTS:
    print("[PREFLIGHT]",r,"exists=",r.exists(),flush=True)
    if not r.exists(): raise FileNotFoundError(r)
    print("[PREFLIGHT]",r.name,len(list(r.rglob("*.avi"))),"AVI",flush=True)

[PREFLIGHT] /kaggle/input/datasets/loliwibu/crosswalkpart-1 exists= True
[PREFLIGHT] crosswalkpart-1 40 AVI
[PREFLIGHT] /kaggle/input/datasets/loliwibu/crosswalk-part-2 exists= True
[PREFLIGHT] crosswalk-part-2 40 AVI
[PREFLIGHT] /kaggle/input/datasets/loliwibu/crosswalk-part-3 exists= True
[PREFLIGHT] crosswalk-part-3 40 AVI


In [3]:
import os,sys,time,json,random,shutil,warnings,subprocess,importlib.util
warnings.filterwarnings("ignore")
from pathlib import Path

def log(x): print(f"[{time.strftime('%H:%M:%S')}] {x}",flush=True)
def ensure(pkg,mod):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

ensure("ultralytics>=8.3.0","ultralytics")

import cv2, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from ultralytics import YOLO

SEED=42
random.seed(SEED); np.random.seed(SEED)

OUT=Path("/kaggle/working/crosswalk_safety_v3_output")
REP=OUT/"reports"; ASSET=OUT/"assets"; FRAMES=ASSET/"annotated_frames"
for p in [OUT,REP,ASSET,FRAMES]: p.mkdir(parents=True,exist_ok=True)

SAMPLE_EVERY_SECONDS=2.0
EVENT_MAX_GAP_SECONDS=4.0
CONF=0.25; IOU=0.50; IMGSZ=640; MAX_DET=300
MAX_SAVE_CONFLICT=120; MAX_SAVE_NORMAL=30
DEVICE=0 if torch.cuda.is_available() else "cpu"

log(f"[CONFIG] device={DEVICE} sample_every={SAMPLE_EVERY_SECONDS}s")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 kB 4.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[23:37:57] [CONFIG] device=0 sample_every=2.0s


In [4]:
video_files=[]
for root in ROOTS:
    vids=sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in {".avi",".mp4",".mov",".mkv"}])
    log(f"[AUDIT] {root.name}: {len(vids)}")
    video_files.extend(vids)
video_files=sorted(video_files,key=lambda p:p.name.lower())

assert len(video_files)==120,f"Expected 120 videos, found {len(video_files)}"

audit=[]
for i,p in enumerate(video_files,1):
    cap=cv2.VideoCapture(str(p))
    opened=cap.isOpened()
    fps=float(cap.get(cv2.CAP_PROP_FPS)) if opened else 0.0
    n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if opened else 0
    w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) if opened else 0
    h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) if opened else 0
    ok,_=cap.read() if opened else (False,None)
    cap.release()
    audit.append({"video":p.name,"path":str(p),"opened":opened,"first_frame_readable":ok,
                  "fps":fps,"frame_count":n,"width":w,"height":h,
                  "duration_seconds":n/fps if fps>0 else 0})
    if i==1 or i%10==0 or i==120:
        log(f"[AUDIT] {i}/120 {p.name} opened={opened} frame={ok}")

audit_df=pd.DataFrame(audit)
audit_df.to_csv(REP/"video_audit.csv",index=False)
bad=audit_df[(~audit_df.opened)|(~audit_df.first_frame_readable)]
assert len(bad)==0,f"Unreadable videos: {len(bad)}"
log("[AUDIT] all 120 videos readable")

[23:37:57] [AUDIT] crosswalkpart-1: 40
[23:37:57] [AUDIT] crosswalk-part-2: 40
[23:37:57] [AUDIT] crosswalk-part-3: 40
[23:37:57] [AUDIT] 1/120 video_001.avi opened=True frame=True
[23:37:58] [AUDIT] 10/120 video_010.avi opened=True frame=True
[23:37:58] [AUDIT] 20/120 video_020.avi opened=True frame=True
[23:37:58] [AUDIT] 30/120 video_030.avi opened=True frame=True
[23:37:59] [AUDIT] 40/120 video_040.avi opened=True frame=True
[23:37:59] [AUDIT] 50/120 video_050.avi opened=True frame=True
[23:38:00] [AUDIT] 60/120 video_060.avi opened=True frame=True
[23:38:00] [AUDIT] 70/120 video_070.avi opened=True frame=True
[23:38:01] [AUDIT] 80/120 video_080.avi opened=True frame=True
[23:38:02] [AUDIT] 90/120 video_090.avi opened=True frame=True
[23:38:02] [AUDIT] 100/120 video_100.avi opened=True frame=True
[23:38:03] [AUDIT] 110/120 video_110.avi opened=True frame=True
[23:38:04] [AUDIT] 120/120 video_120.avi opened=True frame=True
[23:38:04] [AUDIT] all 120 videos readable


In [5]:
model=YOLO("yolo11n.pt")
PERSON=0
VEHICLES={2:"car",3:"motorcycle",5:"bus",7:"truck"}
TARGET=[PERSON]+list(VEHICLES)

DEFAULT_ROI=np.array([[.12,.62],[.88,.62],[.98,.96],[.02,.96]],dtype=np.float32)
ROI_OVERRIDES={}  # optional: {"video_001.avi":[[x,y],...]}

def roi_px(video,w,h):
    pts=np.asarray(ROI_OVERRIDES.get(video,DEFAULT_ROI),dtype=np.float32).copy()
    pts[:,0]*=w; pts[:,1]*=h
    return pts.astype(np.int32)

def inside(pt,poly):
    return cv2.pointPolygonTest(poly.astype(np.float32),(float(pt[0]),float(pt[1])),False)>=0

def annotate(frame,dets,poly,vz,pz,conflict):
    out=frame.copy(); overlay=out.copy()
    cv2.fillPoly(overlay,[poly.reshape(-1,1,2)],(0,255,255))
    out=cv2.addWeighted(overlay,.12,out,.88,0)
    cv2.polylines(out,[poly.reshape(-1,1,2)],True,(0,255,255),3)
    for d in dets:
        x1,y1,x2,y2=map(int,d["xyxy"])
        color=(0,0,255) if d["vehicle"] and d["inside"] else ((0,255,0) if d["person"] and d["inside"] else (255,120,0))
        cv2.rectangle(out,(x1,y1),(x2,y2),color,2)
        cv2.putText(out,f"{d['name']} {d['conf']:.2f}",(x1,max(18,y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX,.55,color,2,cv2.LINE_AA)
    cv2.rectangle(out,(8,8),(900,50),(0,0,0),-1)
    cv2.putText(out,f"vehicle_roi={vz} | ped_roi={pz} | conflict={int(conflict)}",
                (18,38),cv2.FONT_HERSHEY_SIMPLEX,.78,(255,255,255),2,cv2.LINE_AA)
    return out

log("[MODEL] YOLO11n ready")

[23:38:04] [MODEL] YOLO11n ready


In [6]:
frame_rows=[]
saved_conflict=saved_normal=0
global_start=time.time()

for vi,path in enumerate(video_files,1):
    cap=cv2.VideoCapture(str(path))
    fps=float(cap.get(cv2.CAP_PROP_FPS))
    n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if fps<=0: fps=25.0
    step=max(1,int(round(fps*SAMPLE_EVERY_SECONDS)))
    poly=roi_px(path.name,w,h)
    frame_idx=0; sampled=0; conflicts=0

    log(f"[INFER][{vi:03d}/120] START {path.name} fps={fps:.2f} step={step}")

    while True:
        ok=cap.grab()
        if not ok: break
        if frame_idx%step!=0:
            frame_idx+=1; continue
        ok,frame=cap.retrieve()
        if not ok:
            frame_idx+=1; continue

        t=frame_idx/fps
        res=model.predict(frame,imgsz=IMGSZ,conf=CONF,iou=IOU,classes=TARGET,
                          max_det=MAX_DET,device=DEVICE,verbose=False)[0]

        dets=[]; persons=vehicles=pz=vz=0
        if res.boxes is not None:
            for box in res.boxes:
                cls=int(box.cls[0].item()); cf=float(box.conf[0].item())
                xy=box.xyxy[0].cpu().tolist(); x1,y1,x2,y2=xy
                is_p=cls==PERSON; is_v=cls in VEHICLES
                persons+=int(is_p); vehicles+=int(is_v)
                anchor=((x1+x2)/2.0,y2)
                inn=inside(anchor,poly)
                pz+=int(is_p and inn); vz+=int(is_v and inn)
                dets.append({"name":res.names.get(cls,str(cls)),"conf":cf,"xyxy":xy,
                             "person":is_p,"vehicle":is_v,"inside":inn})

        conflict=bool(vz>0 and pz>0)
        sampled+=1; conflicts+=int(conflict)
        frame_rows.append({"video":path.name,"frame_index":frame_idx,"timestamp_seconds":t,
                           "persons_total":persons,"vehicles_total":vehicles,
                           "pedestrians_in_zone":pz,"vehicles_in_zone":vz,
                           "vehicle_presence_in_zone":bool(vz>0),
                           "pedestrian_presence_in_zone":bool(pz>0),
                           "conflict_frame":conflict})

        save=False; kind=""
        if conflict and saved_conflict<MAX_SAVE_CONFLICT:
            save=True; kind="conflict"; saved_conflict+=1
        elif (not conflict) and saved_normal<MAX_SAVE_NORMAL:
            save=True; kind="normal"; saved_normal+=1
        if save:
            img=annotate(frame,dets,poly,vz,pz,conflict)
            cv2.imwrite(str(FRAMES/f"{kind}_{path.stem}_f{frame_idx:07d}_t{t:08.2f}.jpg"),img)

        if sampled==1 or sampled%50==0:
            log(f"[INFER][{path.name}] sampled={sampled} conflicts={conflicts} t={t:.1f}s")
        frame_idx+=1

    cap.release()
    pd.DataFrame(frame_rows).to_csv(REP/"sampled_frame_results.csv",index=False)
    log(f"[INFER][{vi:03d}/120] END {path.name} sampled={sampled} conflicts={conflicts}")

log(f"[INFER] COMPLETE elapsed={(time.time()-global_start)/60:.1f}m")

[23:38:04] [INFER][001/120] START video_001.avi fps=30.00 step=60
[23:38:13] [INFER][video_001.avi] sampled=1 conflicts=0 t=0.0s
[23:38:20] [INFER][video_001.avi] sampled=50 conflicts=2 t=98.0s
[23:38:25] [INFER][video_001.avi] sampled=100 conflicts=9 t=198.0s
[23:38:31] [INFER][video_001.avi] sampled=150 conflicts=12 t=298.0s
[23:38:31] [INFER][001/120] END video_001.avi sampled=150 conflicts=12
[23:38:31] [INFER][002/120] START video_002.avi fps=30.00 step=60
[23:38:31] [INFER][video_002.avi] sampled=1 conflicts=0 t=0.0s
[23:38:38] [INFER][video_002.avi] sampled=50 conflicts=11 t=98.0s
[23:38:45] [INFER][video_002.avi] sampled=100 conflicts=31 t=198.0s
[23:38:52] [INFER][video_002.avi] sampled=150 conflicts=57 t=298.0s
[23:38:52] [INFER][002/120] END video_002.avi sampled=150 conflicts=57
[23:38:52] [INFER][003/120] START video_003.avi fps=30.00 step=60
[23:38:52] [INFER][video_003.avi] sampled=1 conflicts=0 t=0.0s
[23:38:58] [INFER][video_003.avi] sampled=50 conflicts=17 t=98.0s
[23

[mpeg1video @ 0x29f472c0] ac-tex damaged at 73 35
[mpeg1video @ 0x29f472c0] Warning MVs not available


[00:30:00] [INFER][video_108.avi] sampled=100 conflicts=10 t=198.0s
[00:30:13] [INFER][video_108.avi] sampled=150 conflicts=13 t=298.0s
[00:30:13] [INFER][108/120] END video_108.avi sampled=150 conflicts=13
[00:30:13] [INFER][109/120] START video_109.avi fps=30.00 step=60
[00:30:13] [INFER][video_109.avi] sampled=1 conflicts=0 t=0.0s
[00:30:29] [INFER][video_109.avi] sampled=50 conflicts=4 t=98.0s
[00:30:40] [INFER][video_109.avi] sampled=100 conflicts=8 t=198.0s
[00:30:54] [INFER][video_109.avi] sampled=150 conflicts=9 t=298.0s
[00:30:54] [INFER][109/120] END video_109.avi sampled=150 conflicts=9
[00:30:54] [INFER][110/120] START video_110.avi fps=30.00 step=60
[00:30:54] [INFER][video_110.avi] sampled=1 conflicts=0 t=0.0s
[00:31:09] [INFER][video_110.avi] sampled=50 conflicts=3 t=98.0s
[00:31:21] [INFER][video_110.avi] sampled=100 conflicts=4 t=198.0s
[00:31:33] [INFER][video_110.avi] sampled=150 conflicts=7 t=298.0s
[00:31:33] [INFER][110/120] END video_110.avi sampled=150 conflicts

In [7]:
frames_df=pd.DataFrame(frame_rows)

events=[]
for video,g in frames_df.groupby("video"):
    cg=g[g.conflict_frame].sort_values("timestamp_seconds")
    if len(cg)==0: continue
    current=[]; event_id=0; prev=None

    def flush(group,eid):
        if not group: return
        d=pd.DataFrame(group)
        events.append({"video":video,"event_id":eid,
                       "start_seconds":float(d.timestamp_seconds.min()),
                       "end_seconds":float(d.timestamp_seconds.max()),
                       "duration_seconds":float(d.timestamp_seconds.max()-d.timestamp_seconds.min()+SAMPLE_EVERY_SECONDS),
                       "sampled_conflict_frames":int(len(d)),
                       "max_vehicles_in_zone":int(d.vehicles_in_zone.max()),
                       "max_pedestrians_in_zone":int(d.pedestrians_in_zone.max())})

    for _,r in cg.iterrows():
        t=float(r.timestamp_seconds)
        if prev is None or t-prev<=EVENT_MAX_GAP_SECONDS:
            current.append(r.to_dict())
        else:
            flush(current,event_id); event_id+=1; current=[r.to_dict()]
        prev=t
    flush(current,event_id)

events_df=pd.DataFrame(events)
events_df.to_csv(REP/"conflict_events.csv",index=False)

summary_rows=[]
for video,g in frames_df.groupby("video"):
    ev=events_df[events_df.video==video] if len(events_df) else pd.DataFrame()
    summary_rows.append({"video":video,"sampled_frames":len(g),
                         "vehicle_presence_frames":int(g.vehicle_presence_in_zone.sum()),
                         "pedestrian_presence_frames":int(g.pedestrian_presence_in_zone.sum()),
                         "conflict_frames":int(g.conflict_frame.sum()),
                         "conflict_frame_rate":float(g.conflict_frame.mean()),
                         "conflict_events":int(len(ev)),
                         "max_vehicles_in_zone":int(g.vehicles_in_zone.max()),
                         "max_pedestrians_in_zone":int(g.pedestrians_in_zone.max())})

video_summary=pd.DataFrame(summary_rows)
video_summary.to_csv(REP/"video_summary.csv",index=False)
log(f"[EVENT] sampled={len(frames_df)} conflict_frames={int(frames_df.conflict_frame.sum())} grouped_events={len(events_df)}")

[00:41:55] [EVENT] sampled=18000 conflict_frames=6202 grouped_events=1552


In [8]:
plt.figure(figsize=(10,5))
plt.hist(video_summary.conflict_frame_rate,bins=20)
plt.xlabel("Conflict-frame rate"); plt.ylabel("Videos")
plt.title("Crosswalk Safety v3 — Conflict Rate")
plt.tight_layout(); plt.savefig(ASSET/"conflict_rate_distribution.png",dpi=160); plt.close()

top=video_summary.sort_values(["conflict_events","conflict_frames"],ascending=False).head(20)
plt.figure(figsize=(12,6))
plt.bar(top.video,top.conflict_events)
plt.xticks(rotation=75,ha="right"); plt.ylabel("Grouped conflict events")
plt.title("Top Videos by Grouped Conflict Events")
plt.tight_layout(); plt.savefig(ASSET/"top_conflict_videos.png",dpi=160); plt.close()

summary={"project":"8% — Crosswalk Safety","version":"v3",
         "videos_total":len(video_files),"dataset_parts":3,
         "sample_every_seconds":SAMPLE_EVERY_SECONDS,
         "sampled_frames_total":len(frames_df),
         "conflict_frames":int(frames_df.conflict_frame.sum()),
         "conflict_events_grouped":int(len(events_df)),
         "model":"YOLO11n pretrained COCO detector",
         "event_definition":"vehicle and pedestrian both present inside configured demo ROI on same sampled frame; adjacent conflict frames grouped temporally",
         "claim_limit":"Not a validated traffic-violation detector; dataset has no crosswalk polygon ground truth."}
(OUT/"final_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2),flush=True)

zip_path=shutil.make_archive("/kaggle/working/crosswalk_safety_8pct_v3_FULL_VIDEO_outputs","zip",root_dir=OUT)
log(f"[ZIP] {zip_path}")
log("[DONE] CROSSWALK SAFETY v3 PIPELINE FINISHED")

{
  "project": "8% \u2014 Crosswalk Safety",
  "version": "v3",
  "videos_total": 120,
  "dataset_parts": 3,
  "sample_every_seconds": 2.0,
  "sampled_frames_total": 18000,
  "conflict_frames": 6202,
  "conflict_events_grouped": 1552,
  "model": "YOLO11n pretrained COCO detector",
  "event_definition": "vehicle and pedestrian both present inside configured demo ROI on same sampled frame; adjacent conflict frames grouped temporally",
  "claim_limit": "Not a validated traffic-violation detector; dataset has no crosswalk polygon ground truth."
}
[00:41:59] [ZIP] /kaggle/working/crosswalk_safety_8pct_v3_FULL_VIDEO_outputs.zip
[00:41:59] [DONE] CROSSWALK SAFETY v3 PIPELINE FINISHED
